# 20. SVR 시도

코드공유 1등 글이 트리 모델을 아예 안 쓰고 SVR을 씀. raw14+bmi에 그대로 적용해봄.

In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv')
train = train.drop_duplicates(subset=[c for c in train.columns if c != 'ID']).reset_index(drop=True)
train = train.fillna('Unknown')
train['bmi'] = train['weight'].astype(float) / ((train['height'].astype(float) / 100) ** 2)

RAW14_BMI = ['gender', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
             'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity',
             'smoke_status', 'medical_history', 'family_medical_history',
             'sleep_pattern', 'edu_level', 'bmi']
CAT_COLS = ['gender', 'activity', 'smoke_status', 'medical_history',
            'family_medical_history', 'sleep_pattern', 'edu_level']

def make_xy(include_age_mw):
    feats = list(RAW14_BMI)
    if include_age_mw:
        feats += ['age', 'mean_working']
    d = train[feats].copy()
    for c in CAT_COLS:
        dummies = pd.get_dummies(d[c], prefix=c, dtype=int)
        d = pd.concat([d.drop(columns=[c]), dummies], axis=1)
    return d, train['stress_score']

def cv_mae(x, y, C, gamma, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    maes = []
    for tr_idx, va_idx in kf.split(x):
        pipe = make_pipeline(
            RobustScaler(),
            TransformedTargetRegressor(
                regressor=SVR(C=C, gamma=gamma, kernel='rbf', epsilon=0.0),
                transformer=QuantileTransformer(output_distribution='normal',
                                                  n_quantiles=min(1000, len(tr_idx)))
            )
        )
        pipe.fit(x.iloc[tr_idx], y.iloc[tr_idx])
        maes.append(mean_absolute_error(y.iloc[va_idx], pipe.predict(x.iloc[va_idx])))
    return np.mean(maes)

x14, y14 = make_xy(False)
x16, y16 = make_xy(True)
print('raw14+bmi:', cv_mae(x14, y14, 3.96, 1.06))
print('raw14+bmi+age+mean_working:', cv_mae(x16, y16, 3.96, 1.06))

raw14+bmi: 0.15130374638328167
raw14+bmi+age+mean_working: 0.15493300928290024


SVR에서도 age, mean_working 빼는 쪽이 나음. 하이퍼파라미터도 다시 찾아봄.

In [2]:
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

def objective(params):
    return {'loss': cv_mae(x14, y14, params['C'], params['gamma']), 'status': STATUS_OK}

space = {
    'C': hp.loguniform('C', np.log(0.5), np.log(20)),
    'gamma': hp.loguniform('gamma', np.log(0.05), np.log(5)),
}
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=50, trials=trials,
            rstate=np.random.default_rng(RANDOM_STATE))
print('best:', best)
print('best MAE:', min(t['result']['loss'] for t in trials.trials))

best: {'C': 3.8944338291361977, 'gamma': 2.495273322374727}
best MAE: 0.15045845438862615


0.1505. 지금까지 통틀어 제일 좋음 (LGBM 0.1616보다 훨씬 나음).

LGBM이랑 섞으면 더 좋아지는지 확인

In [3]:
import json
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder

train_raw = pd.read_csv('../data/train.csv')
train_raw = train_raw.drop_duplicates(subset=[c for c in train_raw.columns if c != 'ID']).reset_index(drop=True)

lgbm_df = train_raw.copy()
for col in ['medical_history', 'family_medical_history']:
    lgbm_df[col] = lgbm_df[col].fillna('None')
lgbm_df['edu_level'] = lgbm_df['edu_level'].fillna('Unknown')
lgbm_df['bmi'] = lgbm_df['weight'].astype(float) / ((lgbm_df['height'].astype(float) / 100) ** 2)
lgbm_cols = RAW14_BMI + ['stress_score']

with open('optuna_round2_best_params.json') as f:
    lgbm_params = json.load(f)

SVR_C, SVR_GAMMA = best['C'], best['gamma']
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
splits = list(kf.split(train_raw))

oof_svr = np.zeros(len(train_raw))
oof_lgbm = np.zeros(len(train_raw))

for tr_idx, va_idx in splits:
    pipe = make_pipeline(
        RobustScaler(),
        TransformedTargetRegressor(
            regressor=SVR(C=SVR_C, gamma=SVR_GAMMA, kernel='rbf', epsilon=0.0),
            transformer=QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(tr_idx)))
        )
    )
    pipe.fit(x14.iloc[tr_idx], y14.iloc[tr_idx])
    oof_svr[va_idx] = pipe.predict(x14.iloc[va_idx])

    tr_df, va_df = lgbm_df[lgbm_cols].iloc[tr_idx].copy(), lgbm_df[lgbm_cols].iloc[va_idx].copy()
    combo_tr = tr_df['medical_history'] + '_' + tr_df['family_medical_history']
    gmean = tr_df['stress_score'].mean()
    means = tr_df.groupby(combo_tr)['stress_score'].mean()
    tr_df['disease_combo_te'] = combo_tr.map(means).fillna(gmean)
    combo_va = va_df['medical_history'] + '_' + va_df['family_medical_history']
    va_df['disease_combo_te'] = combo_va.map(means).fillna(gmean)
    tr_df = tr_df.drop(columns=['medical_history', 'family_medical_history'])
    va_df = va_df.drop(columns=['medical_history', 'family_medical_history'])
    for c in ['gender', 'activity', 'smoke_status', 'sleep_pattern', 'edu_level']:
        le = LabelEncoder().fit(tr_df[c])
        tr_df[c] = le.transform(tr_df[c])
        unseen = [l for l in np.unique(va_df[c]) if l not in le.classes_]
        if unseen:
            le.classes_ = np.append(le.classes_, unseen)
        va_df[c] = le.transform(va_df[c])
    x_tr = tr_df.drop(columns=['stress_score']); y_tr = tr_df['stress_score']
    x_va = va_df.drop(columns=['stress_score']); y_va = va_df['stress_score']
    m = LGBMRegressor(**lgbm_params)
    m.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgbm[va_idx] = m.predict(x_va)

print('SVR:', mean_absolute_error(y14, oof_svr))
print('LGBM:', mean_absolute_error(y14, oof_lgbm))
for w in [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]:
    blend = w * oof_svr + (1 - w) * oof_lgbm
    print(f'SVR 비중 {w}:', mean_absolute_error(y14, blend))

SVR: 0.1505
LGBM: 0.1616
SVR 비중 0.0: 0.1616
SVR 비중 0.3: 0.1569
SVR 비중 0.5: 0.1544
SVR 비중 0.7: 0.1525
SVR 비중 0.9: 0.1510
SVR 비중 1.0: 0.1505


섞을수록 계속 나빠짐. SVR 단독(비중 1.0)이 제일 좋음. LGBM은 그냥 안 씀.

In [4]:
import os

test = pd.read_csv('../data/test.csv')
test = test.fillna('Unknown')
test['bmi'] = test['weight'].astype(float) / ((test['height'].astype(float) / 100) ** 2)

n_train = len(train)
combined = pd.concat([train[RAW14_BMI], test[RAW14_BMI]], axis=0, ignore_index=True)
for c in CAT_COLS:
    dummies = pd.get_dummies(combined[c], prefix=c, dtype=int)
    combined = pd.concat([combined.drop(columns=[c]), dummies], axis=1)

x_train = combined.iloc[:n_train].reset_index(drop=True)
x_test = combined.iloc[n_train:].reset_index(drop=True)
y_train = train['stress_score']

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(x_train))
test_pred = np.zeros(len(x_test))

for tr_idx, va_idx in kf.split(x_train):
    pipe = make_pipeline(
        RobustScaler(),
        TransformedTargetRegressor(
            regressor=SVR(C=SVR_C, gamma=SVR_GAMMA, kernel='rbf', epsilon=0.0),
            transformer=QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(tr_idx)))
        )
    )
    pipe.fit(x_train.iloc[tr_idx], y_train.iloc[tr_idx])
    oof[va_idx] = pipe.predict(x_train.iloc[va_idx])
    test_pred += pipe.predict(x_test) / kf.n_splits

print('CV:', mean_absolute_error(y_train, np.clip(oof, 0, 1)))

sample_submission = pd.read_csv('../data/sample_submission.csv')
os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = np.clip(test_pred, 0, 1)
sample_submission.to_csv('../submissions/submit_20_svr_breakthrough.csv', index=False)
print('저장 완료')

CV: 0.1505
저장 완료
